# OptiCell Stage 2 — CTC time-lapse (Colab)

**QC → segment → features → phenotype → tracking** on Cell Tracking Challenge 2D sequences, **one dataset at a time**.

| Dataset | Zip | Sequences |
|---------|-----|-----------|
| Fluo-N2DH-GOWT1 | ~53 MB | 01, 02 |
| Fluo-N2DH-SIM+ | ~91 MB | 01, 02 |
| Fluo-N2DL-HeLa | ~182 MB | 01, 02 |

**Fixes included (pull latest `main`):**
- Tracking no longer crashes when the gated cost matrix is all-`inf` (`linear_sum_assignment` infeasible).
- Default track gate `max_distance_px=50` (nuclei can move more than 30 px between frames).

**Policy:** measured outputs only. Cite CTC if you publish.

Runtime: **CPU is enough** for `--backend threshold`.

## 0. Setup — always pull latest before runs

In [ ]:
import os, sys, zipfile, urllib.request, json, shutil, subprocess
from pathlib import Path

USE_DRIVE = False  # True = persist under Google Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/opticell_stage2')
else:
    BASE = Path('/content/opticell_stage2')

BASE.mkdir(parents=True, exist_ok=True)
print('BASE =', BASE)

In [ ]:
REPO = Path('/content/Virelion-OptiCell')
if not REPO.exists():
    !git clone https://github.com/Virelion-Biotech/Virelion-OptiCell.git
else:
    !git -C /content/Virelion-OptiCell fetch origin main
    !git -C /content/Virelion-OptiCell reset --hard origin/main

%cd /content/Virelion-OptiCell
!pip install -e . -q

# Sanity: tracking fix present
from tracking import _assignment
import inspect
src = inspect.getsource(_assignment)
assert 'infeasible' in src or 'finite_mask' in src or 'penalty' in src, (
    'Old tracking.py still loaded — Runtime > Restart session, then re-run this cell'
)
print('OptiCell ready @', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

## 1. Dataset catalog

In [ ]:
DATASETS = {
    'Fluo-N2DH-GOWT1': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-GOWT1.zip',
        'mb': 53,
        'sequences': ['01', '02'],
    },
    'Fluo-N2DH-SIM+': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-SIM+.zip',
        'mb': 91,
        'sequences': ['01', '02'],
    },
    'Fluo-N2DL-HeLa': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DL-HeLa.zip',
        'mb': 182,
        'sequences': ['01', '02'],
    },
}

CTC_ROOT = BASE / 'ctc'
CTC_ROOT.mkdir(parents=True, exist_ok=True)

def download_and_extract(name: str) -> Path:
    meta = DATASETS[name]
    dest_dir = CTC_ROOT / name
    zip_path = CTC_ROOT / f'{name}.zip'
    if dest_dir.exists() and any(dest_dir.iterdir()):
        print(f'[skip download] {name} already at {dest_dir}')
        return dest_dir
    print(f'[download] {name} (~{meta["mb"]} MB) ...')
    urllib.request.urlretrieve(meta['url'], zip_path)
    print(f'[unzip] {zip_path}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(CTC_ROOT)
    if not dest_dir.exists():
        cands = [p for p in CTC_ROOT.iterdir() if p.is_dir() and name.replace('+', '') in p.name.replace('+', '')]
        if cands:
            return cands[0]
        raise FileNotFoundError(f'Extracted folder missing for {name}')
    print(f'[ok] {dest_dir}')
    return dest_dir

print('Datasets:', list(DATASETS))

## 2. Stage-2 runner (threshold + tracking)

In [ ]:
def run_stage2(
    dataset_name: str,
    sequence: str,
    max_frames: int = 0,
    track_max_distance: float = 50.0,
    track_max_gap: int = 1,
):
    root = download_and_extract(dataset_name)
    seq_dir = root / sequence
    if not seq_dir.is_dir():
        alt = root / dataset_name / sequence
        seq_dir = alt if alt.is_dir() else seq_dir
    if not seq_dir.is_dir():
        print('Under', root, ':', list(root.iterdir())[:30])
        raise FileNotFoundError(seq_dir)

    frames = [f for f in sorted(seq_dir.glob('*.tif')) + sorted(seq_dir.glob('*.tiff'))
              if not f.name.startswith('._')]
    print(f'{dataset_name}/{sequence}: {len(frames)} frames in {seq_dir}')
    if not frames:
        raise RuntimeError('no frames')

    out = BASE / 'outputs' / f'stage2_{dataset_name}_{sequence}'
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        str(REPO / 'scripts' / 'run_killer_workflow.py'),
        str(seq_dir),
        '-o', str(out),
        '--backend', 'threshold',
        '--enable-tracking',
        '--track-max-distance', str(track_max_distance),
        '--track-max-gap', str(track_max_gap),
    ]
    if max_frames and max_frames > 0:
        cmd += ['--max-images', str(max_frames)]

    print('CMD:', ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    # show tail of stdout always
    if r.stdout:
        print(r.stdout[-5000:])
    if r.returncode != 0:
        print('STDERR tail:', (r.stderr or '')[-3000:])
        raise RuntimeError(f'Stage2 failed code={r.returncode}')

    summary_path = out / 'workflow_summary.json'
    if not summary_path.exists():
        raise RuntimeError(f'missing {summary_path}')
    summary = json.loads(summary_path.read_text())
    print('=== SUMMARY', dataset_name, sequence, '===')
    for k, v in summary.get('summary', {}).items():
        print(f'  {k}: {v}')
    return summary

print('run_stage2() ready')

## 3. Run one-by-one

`MAX_FRAMES = 0` → full sequence. Use `20` for a smoke test.

If a cell fails after a code fix: re-run **section 0 install cell**, then re-run the failed dataset cell.

In [ ]:
MAX_FRAMES = 0
TRACK_DIST = 50.0
TRACK_GAP = 1

# --- 1/6 GOWT1 / 01 ---
s1 = run_stage2('Fluo-N2DH-GOWT1', '01', max_frames=MAX_FRAMES,
               track_max_distance=TRACK_DIST, track_max_gap=TRACK_GAP)

In [ ]:
# --- 2/6 GOWT1 / 02 ---
s2 = run_stage2('Fluo-N2DH-GOWT1', '02', max_frames=MAX_FRAMES,
               track_max_distance=TRACK_DIST, track_max_gap=TRACK_GAP)

In [ ]:
# --- 3/6 SIM+ / 01 ---
s3 = run_stage2('Fluo-N2DH-SIM+', '01', max_frames=MAX_FRAMES,
               track_max_distance=TRACK_DIST, track_max_gap=TRACK_GAP)

In [ ]:
# --- 4/6 SIM+ / 02 ---
s4 = run_stage2('Fluo-N2DH-SIM+', '02', max_frames=MAX_FRAMES,
               track_max_distance=TRACK_DIST, track_max_gap=TRACK_GAP)

In [ ]:
# --- 5/6 HeLa / 01 ---
s5 = run_stage2('Fluo-N2DL-HeLa', '01', max_frames=MAX_FRAMES,
               track_max_distance=TRACK_DIST, track_max_gap=TRACK_GAP)

In [ ]:
# --- 6/6 HeLa / 02 ---
s6 = run_stage2('Fluo-N2DL-HeLa', '02', max_frames=MAX_FRAMES,
               track_max_distance=TRACK_DIST, track_max_gap=TRACK_GAP)

## 4. Aggregate measured summaries

In [ ]:
import pandas as pd

rows = []
out_root = BASE / 'outputs'
for p in sorted(out_root.glob('stage2_*/workflow_summary.json')):
    data = json.loads(p.read_text())
    s = data.get('summary', {})
    rows.append({
        'run': p.parent.name,
        'backend': data.get('backend'),
        'n_images': s.get('n_images'),
        'mean_object_count': s.get('mean_object_count'),
        'mean_confidence': s.get('mean_confidence'),
        'mean_focus': s.get('mean_focus'),
        'n_objects_total': s.get('n_objects_total'),
        'n_tracks': s.get('n_tracks'),
        'tracking_enabled': s.get('tracking_enabled'),
        'phenotype_positive_fraction': s.get('phenotype_positive_fraction'),
    })

table = pd.DataFrame(rows)
display(table)
agg_path = out_root / 'stage2_ctc_aggregate.csv'
table.to_csv(agg_path, index=False)
print('Wrote', agg_path)
print('Paste this table back for Stage-2 docs — measured only, no invented TRA scores.')

## 5. Zip outputs (optional)

In [ ]:
zip_out = '/content/stage2_ctc_outputs.zip'
!cd {BASE} && zip -r -q {zip_out} outputs
print('Created', zip_out)
# from google.colab import files
# files.download(zip_out)

### Cite

- Cell Tracking Challenge: https://celltrackingchallenge.net/datasets/
- OptiCell: https://github.com/Virelion-Biotech/Virelion-OptiCell